# Datatvätt och transformering
1. Utforskning av data
2. Transformering till tabeller
3. Validera data
4. Exportera till csv-filer


## Setup

In [1]:
import pandas as pd
import os
os.makedirs("data", exist_ok=True)

## Utforskning av data

In [2]:
raw = pd.read_json("../data/raw/show_details.jsonl", lines=True)
raw.head()

,id,popularity,name,episode_run_time,number_of_episodes,number_of_seasons,seasons,networks,genres
0,94997,498.4620,House of the Dragon,[],26,3,"[{'season_number': 0, 'air_date': '2022-08-21'...",[{'network': 'HBO'}],"[{'genre': 'Sci-Fi & Fantasy'}, {'genre': 'Dra..."
1,113962,342.4827,Lioness,[],24,3,"[{'season_number': 1, 'air_date': '2023-07-23'...",[{'network': 'Paramount+'}],"[{'genre': 'Drama'}, {'genre': 'War & Politics'}]"
2,5920,316.5748,The Mentalist,[],151,7,"[{'season_number': 0, 'air_date': '2009-08-20'...",[{'network': 'CBS'}],"[{'genre': 'Crime'}, {'genre': 'Drama'}, {'gen..."
3,2734,257.0061,Law & Order: Special Victims Unit,[],595,28,"[{'season_number': 0, 'air_date': None, 'episo...",[{'network': 'NBC'}],"[{'genre': 'Crime'}, {'genre': 'Drama'}, {'gen..."
4,1416,245.1795,Grey's Anatomy,[],466,23,"[{'season_number': 0, 'air_date': '2006-02-14'...",[{'network': 'ABC'}],[{'genre': 'Drama'}]


#### Episode run time
Fråga: Är det så att antalet rader utan episode_run_time gör att det inte går att användas för någon analys.

Svar: ja.

In [3]:
raw[raw["episode_run_time"].apply(lambda x: len(x) == 0)]

,id,popularity,name,episode_run_time,number_of_episodes,number_of_seasons,seasons,networks,genres
0,94997,498.4620,House of the Dragon,[],26,3,"[{'season_number': 0, 'air_date': '2022-08-21'...",[{'network': 'HBO'}],"[{'genre': 'Sci-Fi & Fantasy'}, {'genre': 'Dra..."
1,113962,342.4827,Lioness,[],24,3,"[{'season_number': 1, 'air_date': '2023-07-23'...",[{'network': 'Paramount+'}],"[{'genre': 'Drama'}, {'genre': 'War & Politics'}]"
2,5920,316.5748,The Mentalist,[],151,7,"[{'season_number': 0, 'air_date': '2009-08-20'...",[{'network': 'CBS'}],"[{'genre': 'Crime'}, {'genre': 'Drama'}, {'gen..."
3,2734,257.0061,Law & Order: Special Victims Unit,[],595,28,"[{'season_number': 0, 'air_date': None, 'episo...",[{'network': 'NBC'}],"[{'genre': 'Crime'}, {'genre': 'Drama'}, {'gen..."
4,1416,245.1795,Grey's Anatomy,[],466,23,"[{'season_number': 0, 'air_date': '2006-02-14'...",[{'network': 'ABC'}],[{'genre': 'Drama'}]
...,...,...,...,...,...,...,...,...,...
1178,73000,5.7061,The Miniaturist,[],2,1,"[{'season_number': 1, 'air_date': '2017-12-26'...",[{'network': 'BBC One'}],"[{'genre': 'Mystery'}, {'genre': 'Drama'}]"
1180,49157,5.4267,Houdini,[],2,1,"[{'season_number': 1, 'air_date': '2014-09-01'...",[{'network': 'History'}],[{'genre': 'Drama'}]
1197,100052,4.5330,The English Game,[],6,1,"[{'season_number': 1, 'air_date': '2020-03-20'...",[{'network': 'Netflix'}],[{'genre': 'Drama'}]
1203,129418,4.5841,Brand New Cherry Flavor,[],8,1,"[{'season_number': 1, 'air_date': '2021-08-13'...",[{'network': 'Netflix'}],"[{'genre': 'Drama'}, {'genre': 'Mystery'}, {'g..."


## Skapa tabeller

In [4]:
# --- SEASON-tabell ---
season = raw[["id", "seasons"]].explode("seasons")
season = pd.json_normalize(season["seasons"]).assign(show_id=season["id"].values)
season = season[["show_id", "season_number", "air_date", "episode_count"]]

# Filtrera bort season 0 (extramaterial)
season = season[season["season_number"] != 0]

# Filtrera bort säsonger utan air_date (framtida/oannonserade säsonger)
season = season[season["air_date"].notna() & (season["air_date"] != "")]

# Filtrera bort serier som (efter ovanstående) bara har 1 säsong kvar
season_counts = season.groupby("show_id").size()
valid_show_ids = season_counts[season_counts > 1].index
season = season[season["show_id"].isin(valid_show_ids)]

# --- GENRE-tabell ---
genre = raw[["id", "genres"]].explode("genres")
genre = pd.json_normalize(genre["genres"]).assign(show_id=genre["id"].values)
genre = genre[["show_id", "genre"]].rename(columns={"genre": "genre_name"})
genre = genre[genre["show_id"].isin(valid_show_ids)]

# --- NETWORK-tabell ---
network = raw[["id", "networks"]].explode("networks")
network = pd.json_normalize(network["networks"]).assign(show_id=network["id"].values)
network = network[["show_id", "network"]].rename(columns={"network": "network_name"})
network = network[network["show_id"].isin(valid_show_ids)]

# --- SHOWS-tabell ---
shows = raw[raw["id"].isin(valid_show_ids)][["id", "name", "popularity", "number_of_episodes", "number_of_seasons"]]

## Validera data

In [16]:
# Kolla flera varianter av "tomt" samtidigt
empty_air_dates = season[
    season["air_date"].isna() | (season["air_date"] == "") | (season["air_date"].isnull())
]

print(f"Antal säsonger med tom/saknad air_date: {len(empty_air_dates)}")
empty_air_dates

Antal säsonger med tom/saknad air_date: 0


,show_id,season_number,air_date,episode_count


In [17]:
empty_genres_raw = raw[raw["genres"].apply(len) == 0]
print(f"Antal serier utan genre: {len(empty_genres_raw)}")
empty_genres_raw[["id", "name"]]

Antal serier utan genre: 0


,id,name


In [4]:
def check_empty_values(df, name="DataFrame"):
    print(f"--- Kollar tomma värden i {name} ---")
    for col in df.columns:
        sample = df[col].dropna().iloc[0] if df[col].notna().any() else None

        if isinstance(sample, list):
            # Listkolumn: kolla tomma listor
            empty_count = df[col].apply(lambda x: isinstance(x, list) and len(x) == 0).sum()
        else:
            # Vanlig kolumn: kolla NaN, None och tom sträng
            empty_count = (df[col].isna() | (df[col] == "")).sum()

        if empty_count > 0:
            print(f"  {col}: {empty_count} tomma värden")

    print()


# Använd på dina olika tabeller
check_empty_values(raw, "raw")
check_empty_values(season, "season")
check_empty_values(genre, "genre")
check_empty_values(network, "network")
check_empty_values(shows, "shows")

--- Kollar tomma värden i raw ---
  episode_run_time: 616 tomma värden

--- Kollar tomma värden i season ---

--- Kollar tomma värden i genre ---

--- Kollar tomma värden i network ---

--- Kollar tomma värden i shows ---



## Räkna ut dagar från föregående säsong

In [5]:
# Se till att air_date faktiskt är ett datetime-objekt, inte en sträng
season["air_date"] = pd.to_datetime(season["air_date"])

# Sortera på air_date, inte season_number - finns fall där säsong 2 
# börjar sändas före säsong 1 enligt TMDB:s numrering
season = season.sort_values(["show_id", "air_date"])

# Räkna ut antal dagar sen föregående säsong, per serie
season["days_since_prev_season"] = (
    season.groupby("show_id")["air_date"].diff().dt.days
)

In [22]:
season[season["show_id"] == 60797]

,show_id,season_number,air_date,episode_count,days_since_prev_season
264,60797,1,2014-09-22,22,NaN
264,60797,2,2015-09-21,24,364.0
264,60797,3,2016-10-03,25,378.0
264,60797,4,2017-09-25,22,357.0


In [6]:
shows.to_csv("data/shows.csv", index=False)
season.to_csv("data/season.csv", index=False)
genre.to_csv("data/genre.csv", index=False)
network.to_csv("data/network.csv", index=False)

In [22]:
negative = season[season["days_since_prev_season"] < 0]
negative

,show_id,season_number,air_date,episode_count,days_since_prev_season
532,90,6,2007-12-16,3,-115.0
532,90,8,2009-06-10,2,-147.0


In [25]:
season["days_since_prev_season"].sort_values(ascending=False).head(50)

632     4690.0
1081    4672.0
938     3602.0
898     3360.0
665     3137.0
27      2806.0
1046    2707.0
266     2523.0
345     2518.0
167     2275.0
149     2192.0
27      1829.0
1053    1827.0
1041    1592.0
992     1573.0
246     1553.0
618     1531.0
322     1519.0
31      1484.0
527     1472.0
71      1471.0
995     1465.0
593     1372.0
71      1348.0
5       1329.0
634     1316.0
1053    1302.0
847     1288.0
783     1279.0
882     1274.0
112     1267.0
258     1257.0
949     1250.0
140     1250.0
809     1239.0
703     1174.0
258     1150.0
1040    1146.0
719     1134.0
199     1134.0
46      1120.0
490     1113.0
1031    1112.0
54      1107.0
130     1105.0
8       1100.0
301     1099.0
846     1098.0
741     1096.0
847     1090.0
Name: days_since_prev_season, dtype: float64

In [6]:
season["days_since_prev_season"].describe()

count    3258.000000
mean      449.871700
std       354.534865
min         1.000000
25%       364.000000
50%       371.000000
75%       448.000000
max      9730.000000
Name: days_since_prev_season, dtype: float64

## Hantering av outlier-värden. (Serier som har avslutats helt men senare återkommit)